# Fine-tune Gemma 4 E4B for PCB component placement (Kaggle)

Single T4 + QLoRA. Reads the DSL JSONL from a Kaggle Dataset, trains a LoRA adapter, evaluates against random / net-centroid baselines, and pushes the adapter to a private HF Hub repo.

Before running:
- Settings → Accelerator → **GPU T4 ×2** (we'll only use one of the two).
- Settings → Internet → **on**.
- Settings → Add-ons → Secrets → add `HF_TOKEN` (read+write scope).
- Add-ons → Datasets → attach `fine-tune-pcb-dsl` (your private dataset containing `train.jsonl`).
- Accept Gemma 4 terms once at https://huggingface.co/google/gemma-4-E4B-it on the HF account whose token you're using.

See `docs/kaggle.md` in the repo for the full runbook.

In [ ]:
# 1. Clone the project source.
#
# Use Python os.chdir so we don't depend on the bash subprocess's cwd —
# if a previous run left it pointing at a deleted dir, !-commands inherit
# the dead cwd and fail with "getcwd: cannot access parent directories".
import os
os.chdir("/kaggle/working")
!rm -rf /kaggle/working/fine-tune-pcb
!git clone https://github.com/bpkneale/fine-tune-pcb.git /kaggle/working/fine-tune-pcb
os.chdir("/kaggle/working/fine-tune-pcb")
print("cwd:", os.getcwd())
!ls

In [ ]:
# 2. Install / upgrade deps. TRL's surface keeps changing — pin a floor.
# Cap protobuf<7 because Kaggle's preinstalled google-cloud-* packages
# refuse to coexist with protobuf 7.x. Uninstall wandb because Kaggle's
# preinstalled wandb has proto stubs incompatible with protobuf<7, and
# TRL eagerly imports wandb at module load time via is_wandb_available()
# even when report_to=[] disables the integration.
!pip uninstall -y wandb
!pip install -q -U \
    "transformers>=4.50" "peft>=0.13" "datasets>=2.20" \
    "accelerate>=1.0" "trl>=0.13" "bitsandbytes>=0.43" \
    "liger-kernel>=0.4" sentencepiece "protobuf>=4,<7" orjson tqdm
!pip install -q -e .

In [ ]:
# 3. HF auth from Kaggle Secrets. Pin to the first T4 — we don't need the
# second card and skipping device_map keeps PEFT happy.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
# `huggingface-cli` was renamed to `hf` in huggingface_hub >= 0.27.
!hf auth login --token $HF_TOKEN --add-to-git-credential

In [ ]:
# 4. Tokenise. Full seq_len 4096 — QLoRA gives us the headroom on T4 16GB.
# Verify the dataset mount before invoking, then point at the directory
# (tokenize_dataset.py auto-globs for *.jsonl inside).
!ls /kaggle/input/
!python -m src.tokenize_dataset \
    --in /kaggle/input/fine-tune-pcb-dsl \
    --out /kaggle/working/hf-data \
    --max-tokens 4096

In [ ]:
# 5. Train. QLoRA + Liger Kernel auto-detected. ~2-3 hours on T4.
!python -m src.train_lora \
    --dataset /kaggle/working/hf-data \
    --output-dir /kaggle/working/adapter \
    --epochs 3 \
    --per-device-batch-size 2 \
    --grad-accum 8 \
    --seq-len 4096

In [ ]:
# 6. Held-out eval against random + net-centroid baselines.
!python -m src.eval_placement \
    --adapter /kaggle/working/adapter \
    --dataset /kaggle/working/hf-data \
    --max-examples 200 \
    --out /kaggle/working/eval-results.json
!cat /kaggle/working/eval-results.json

In [ ]:
# 7. Push adapter to a private HF Hub model repo so we can pull it locally.
from huggingface_hub import HfApi, create_repo
REPO_ID = "bpkneale/gemma4-e4b-pcb-lora-v1"
create_repo(REPO_ID, repo_type="model", private=True, exist_ok=True)
HfApi().upload_folder(
    folder_path="/kaggle/working/adapter",
    repo_id=REPO_ID,
    repo_type="model",
)
print(f"pushed to https://huggingface.co/{REPO_ID}")